<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Wekk7_Day3_Exercises_XP_RAG_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [1]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


/tmp/ipykernel_2195/1013419477.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


## 1) Load dataset and convert to Documents


In [3]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
            page_content=row[text_column],
            metadata={"source": row[source_column]}
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

huggingface_doc.csv:   0%|          | 0.00/22.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

Documents: 200
Example: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx'}
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 



## 2) Split into chunks


In [4]:
chunk_size = 512
chunk_overlap = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

splits = splitter.split_documents(documents)
print("Chunks:", len(splits))
print("First chunk metadata:", splits[0].metadata)
print(splits[0].page_content[:350])

Chunks: 5789
First chunk metadata: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx'}
Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 




## 3) Vector store + retriever (FAISS)


In [5]:
from langchain_community.vectorstores import FAISS, DistanceStrategy

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.COSINE
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")

/tmp/ipykernel_2195/2144377066.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retriever ready


### 3.1) Vérification de cohérence de la récupération
Testons l'impact du paramètre `k` sur les documents récupérés.

In [6]:
def test_retrieval(query, k_value):
    # Vérification si vectorstore existe dans l'espace de noms global
    if 'vectorstore' not in globals():
        print("Erreur : La variable 'vectorstore' n'est pas définie. Veuillez exécuter la cellule 70707070 avant cette étape.")
        return

    print(f"\n--- Recherche avec k={k_value} ---")
    temp_retriever = vectorstore.as_retriever(search_kwargs={'k': k_value})
    docs = temp_retriever.invoke(query)
    for i, doc in enumerate(docs):
        print(f"Document {i+1} (Source: {doc.metadata.get('source')}):")
        print(f"{doc.page_content[:200]}...\n")

query_test = "How to deploy a model on Hugging Face?"
test_retrieval(query_test, k_value=2)
test_retrieval(query_test, k_value=6)


--- Recherche avec k=2 ---
Document 1 (Source: huggingface/transformers/blob/main/templates/adding_a_new_model/ADD_NEW_MODEL_PROPOSAL_TEMPLATE.md):
### Overview of tokenizers

Not quite ready yet :-( This section will be added soon!

Step-by-step recipe to add a model to 🤗 Transformers
----------------------------------------------------

Everyon...

Document 2 (Source: huggingface/blog/blob/main/mantis-case-study.md):
- Train model on a GPU instance (provisioned by  [CML](https://cml.dev/), trained with [transformers](https://huggingface.co/docs/transformers/main/))
- Upload to [Hugging Face Hub](https://huggingfac...


--- Recherche avec k=6 ---
Document 1 (Source: huggingface/transformers/blob/main/templates/adding_a_new_model/ADD_NEW_MODEL_PROPOSAL_TEMPLATE.md):
### Overview of tokenizers

Not quite ready yet :-( This section will be added soon!

Step-by-step recipe to add a model to 🤗 Transformers
----------------------------------------------------

Everyon...

Document 2 (Sourc

## 4) Build the RAG chain


In [7]:
from typing import Any, List, Optional
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from langchain_core.language_models.llms import LLM
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# Custom LLM wrapper to handle Seq2Seq correctly in this environment
class FlanT5LLM(LLM):
    model_id: str = "google/flan-t5-small"
    _tokenizer: Any = None
    _model: Any = None

    def __init__(self, model_id, **kwargs):
        super().__init__(**kwargs)
        object.__setattr__(self, "model_id", model_id)
        object.__setattr__(self, "_tokenizer", AutoTokenizer.from_pretrained(model_id))
        object.__setattr__(self, "_model", AutoModelForSeq2SeqLM.from_pretrained(model_id))

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        inputs = self._tokenizer(prompt, return_tensors="pt").to(self._model.device)
        # For Seq2Seq models like T5, generate() returns only the newly generated tokens
        outputs = self._model.generate(**inputs, max_new_tokens=256)
        response = self._tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response

    @property
    def _llm_type(self) -> str:
        return "custom_flan_t5"

template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Question: {question}
Answer:"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

# Initialize our custom LLM
llm = FlanT5LLM(model_id="google/flan-t5-small")

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

print("RAG chain updated with optimized Custom Seq2Seq LLM wrapper")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

RAG chain updated with optimized Custom Seq2Seq LLM wrapper


## 5) Demo: RAG vs no-RAG


In [8]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# No-RAG (LLM only)
no_rag_prompt = f"Answer the question concisely: {q}"
no_rag_answer = llm.invoke(no_rag_prompt)

# RAG
rag_result = qa.invoke(q)

print("Question:", q)
print("\n--- No-RAG Answer ---")
print(no_rag_answer)

print("\n--- RAG Answer ---")
print(rag_result["result"])

print("\nSources:")
for i, d in enumerate(rag_result["source_documents"]):
    print(f"{i+1}. {d.metadata.get('source')}")

Question: How can I retrieve a model from the Hugging Face Hub?

--- No-RAG Answer ---
Use a hammer to hammer the model into the hammer.

--- RAG Answer ---
Create a model repository on the [Hugging Face website](https://huggingface.co/join).

Sources:
1. huggingface/blog/blob/main/fastai.md
2. huggingface/transformers/blob/main/docs/source/en/model_sharing.md
3. huggingface/blog/blob/main/fastai.md
4. huggingface/transformers/blob/main/docs/source/en/model_sharing.md


### Questions de validation du RAG
Voici une série de questions pour tester la précision du modèle Flan-T5 par rapport à la documentation Hugging Face.

In [9]:
questions = [
    "What is a Hugging Face Endpoint?",
    "How do I use RecursiveCharacterTextSplitter?",
    "What is the role of the metadata in LangChain documents?"
]

for query in questions:
    result = qa.invoke(query)
    print(f"\nQuestion: {query}")
    print(f"Réponse: {result['result']}")
    print(f"Source principale: {result['source_documents'][0].metadata.get('source')}")


Question: What is a Hugging Face Endpoint?
Réponse: (iii)
Source principale: huggingface/course/blob/main/README.md

Question: How do I use RecursiveCharacterTextSplitter?
Réponse: Use [SentencePiece]
Source principale: huggingface/course/blob/main/chapters/en/chapter6/4.mdx

Question: What is the role of the metadata in LangChain documents?
Réponse: a vital tool for finding relevant datasets
Source principale: huggingface/blog/blob/main/huggy-lingo.md


```markdown
# 📘 Guide Compréhensif : Architecture du Pipeline RAG

Ce document résume les étapes clés pour construire un système de **Retrieval-Augmented Generation (RAG)** avec LangChain.

### 1. Préparation de l'Environnement
Nous installons les bibliothèques nécessaires :
- `langchain` & `langchain-huggingface` : Le framework pour orchestrer le pipeline.
- `faiss-cpu` : La base de données vectorielle (Vector Store) pour stocker nos connaissances.
- `sentence-transformers` : Pour transformer le texte en vecteurs numériques (Embeddings).
- `transformers` & `datasets` : Pour charger le modèle de langage (LLM) et les données de Hugging Face.

### 2. Chargement des Données (Loading)
Le but est de transformer des données brutes (ici le dataset `huggingface_doc`) en objets `Document` compréhensibles par LangChain.
- **Metadata** : On conserve la source (URL ou chemin) pour pouvoir citer les références lors de la réponse finale.

### 3. Découpage en Tronçons (Splitting)
Un LLM a une limite de mémoire (fenêtre de contexte). On utilise `RecursiveCharacterTextSplitter` :
- **Chunk Size (512)** : Taille maximale d'un morceau.
- **Overlap (50)** : On fait chevaucher les morceaux pour ne pas perdre le sens entre deux paragraphes.

### 4. Création des Embeddings & Vector Store
- **Embeddings** (`all-MiniLM-L6-v2`) : C'est un modèle qui transforme une phrase en une liste de chiffres (vecteur). Deux phrases avec un sens proche auront des vecteurs proches.
- **FAISS** : Stocke ces vecteurs et permet une recherche ultra-rapide par similarité cosinus.

### 5. Le Retriever (Le Documentaliste)
C'est l'interface qui va chercher les `k` documents les plus pertinents dans FAISS lorsqu'on lui pose une question. Il ne génère pas de texte, il sélectionne les meilleures sources.

### 6. Le LLM & La Chaîne RAG
- **Modèle** (`Flan-T5-Small`) : Un modèle encodeur-décodeur qui va lire la question ET le contexte fourni par le retriever pour formuler une réponse.
- **Prompt Template** : C'est la consigne donnée au modèle : *"Utilise uniquement le contexte suivant pour répondre... si tu ne sais pas, dis-le"*.
- **RetrievalQA** : C'est la chaîne qui lie tout ensemble automatiquement.

### 7. Pourquoi le RAG est-il utile ?
Comme vu dans la démo :
- **Sans RAG** : Le modèle hallucine ou donne des réponses génériques (ex: l'histoire du marteau).
- **Avec RAG** : Le modèle base sa réponse sur des faits réels extraits de la documentation, augmentant ainsi la fiabilité et la précision.
```